## Notebook to learn to play with tif images

In [1]:
import sys
import importlib as imp
import numpy as np
import matplotlib.pyplot as plt
import rasterio

import experiment_settings
import build_model
import train_model
import build_data

import tensorflow as tf

# tf.config.set_visible_devices([], "GPU")  # turn-off tensorflow-metal if it is on

In [2]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"tensorflow version = {tf.__version__}")  

print(tf.config.list_physical_devices('GPU'))

python version = 3.10.10 | packaged by conda-forge | (main, Mar 24 2023, 20:12:31) [Clang 14.0.6 ]
numpy version = 1.23.2
tensorflow version = 2.10.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
EXP_NAME = "exp0"
settings = experiment_settings.get_settings(EXP_NAME)
# display(settings)

In [4]:
imp.reload(build_data)

data_generator = build_data.data_generator(settings)

sample_years, sample_lats, sample_lons = build_data.make_sample_list(settings)
print(sample_years.shape, sample_lons.shape, sample_lats.shape)

x_tfds = tf.data.Dataset.from_tensor_slices((sample_years, sample_lats, sample_lons))
y_tfds = tf.data.Dataset.from_tensor_slices((sample_years, sample_lats, sample_lons))

# these should not be split??????
x_tfds = x_tfds.batch(settings["batch_size"]).shuffle(buffer_size = int(len(sample_years)/settings["batch_size"]), reshuffle_each_iteration=True, seed = settings["rng_seed"])
y_tfds = y_tfds.batch(settings["batch_size"]).shuffle(buffer_size = int(len(sample_years)/settings["batch_size"]), reshuffle_each_iteration=True, seed = settings["rng_seed"])

x_tfds = x_tfds.map(lambda sample_years, sample_lats, sample_lons: 
                    tf.py_function(data_generator.get_x_data, [sample_years, sample_lats, sample_lons], 
                                   Tout=tf.float64))

y_tfds = y_tfds.map(lambda sample_years, sample_lats, sample_lons: 
                    tf.py_function(data_generator.get_y_data, [sample_years, sample_lats, sample_lons], 
                                   Tout=tf.float64))

init_batch_x = next(x_tfds.as_numpy_iterator())
tfds_all = tf.data.Dataset.zip((x_tfds, y_tfds))
tfds_all = tfds_all.prefetch(tf.data.AUTOTUNE)

(15683, 36390)
(12800,) (12800,) (12800,)
Metal device set to: Apple M1 Max

systemMemory: 64.00 GB
maxCacheSize: 24.00 GB



KeyboardInterrupt: 

In [ ]:
imp.reload(build_model)
imp.reload(train_model)

# need to look into whether this is actually doing what we think it is given the BATCH/SHUFFLE above
tfds_train = tfds_all.take(settings["n_batches"][0])
tfds_val = tfds_all.skip(settings["n_batches"][0]).take(settings["n_batches"][1])

model = build_model.build_model(settings, input_shape=np.shape(init_batch_x)[1:])
model, fit_summary, history, settings = train_model.train_model(settings, model, tfds_train, tfds_val)